# Figure 5b — Pairwise distance matrix of language date-vectors

Each language is represented as a 6-week-summed frequency vector over time.
The distance matrix shows how similar languages' temporal trajectories are.

**Inputs:** `CHOSEN_WEEKLY_PIVOT_FILE` (`data/processed/chosen_words_weekly_pivoted.csv`)
**Outputs:** `outputs/figures/Fig.5b_Vectors_matrix/<timestamp>/`
**Prerequisites:** run `02_combine_data.ipynb` first; `pip install scikit-learn scipy`

In [ ]:
import sys
sys.path.insert(0, '..')
from config import CHOSEN_WEEKLY_PIVOT_FILE, WORD_FORMS_ALL, FIGURES_DIR

import pandas as pd
import numpy as np
from datetime import datetime
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.colors as mcolors
from mpl_toolkits.axes_grid1 import make_axes_locatable
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.metrics.pairwise import cosine_distances, euclidean_distances
from scipy.spatial.distance import cdist

pio.templates.default = "plotly_white"

## ⚙️ Parameters

In [ ]:
# Distance metric: 'cosine', 'euclidean', or 'jensenshannon'
metric = 'jensenshannon'

## Load data & 6-week aggregation

In [ ]:
DFfreq = pd.read_csv(CHOSEN_WEEKLY_PIVOT_FILE, index_col=0)

# Robust index handling: supports both ISO-index and language-index pivots
meta_lang = pd.read_csv(WORD_FORMS_ALL, usecols=['ISO', 'Language']).drop_duplicates()
iso_to_name = meta_lang.set_index('ISO')['Language'].to_dict()
iso_set = set(meta_lang['ISO'])
name_set = set(meta_lang['Language'])

idx = pd.Index(DFfreq.index.astype(str))
if idx.isin(iso_set).all():
    DFfreq.index = idx.map(iso_to_name)
elif idx.isin(name_set).all():
    DFfreq.index = idx
else:
    DFfreq.index = idx.map(iso_to_name).fillna(idx)
DFfreq.index = pd.Index(DFfreq.index, name='language')

# Keep only date columns, convert to datetime
date_mask = pd.to_datetime(DFfreq.columns, errors='coerce').notna()
DFfreq = DFfreq.loc[:, date_mask].copy()
DFfreq.columns = pd.to_datetime(DFfreq.columns)

# Group into 6-week periods (rows = languages, columns = 6W period end-dates)
df = DFfreq.T.groupby(
    pd.Grouper(freq='6W', label='right', closed='right')
).sum().T

# For the distance matrix each 6W period is one observation, languages are its features
df_transposed = df.T  # shape: (6W-periods × languages)

print(f"Matrix input shape: {df_transposed.shape}  (6W-periods × languages)")
df_transposed.head(3)

## Compute distance matrix

In [ ]:
if metric == 'cosine':
    distance_matrix = cosine_distances(df_transposed.values)
elif metric == 'euclidean':
    distance_matrix = euclidean_distances(df_transposed.values)
elif metric in ('jensenshannon', 'JSD'):
    distance_matrix = cdist(
        df_transposed.values, df_transposed.values, metric='jensenshannon'
    )
else:
    raise ValueError(
        f"metric must be 'cosine', 'euclidean', or 'jensenshannon', got: {metric!r}"
    )

distance_df = pd.DataFrame(
    distance_matrix,
    index=df_transposed.index,
    columns=df_transposed.index,
)
print(f"Distance matrix shape: {distance_df.shape}")
distance_df.head(3)

## Interactive heatmap (Plotly)

In [ ]:
# log10 of distance values for display (diagonal = 0 → -inf shown as min colour)
z_log_interactive = np.log10(np.where(distance_df.values == 0,
                                      np.nan, distance_df.values))

dates_idx   = pd.to_datetime(distance_df.index)
unique_years = sorted(set(dates_idx.year))
tickvals_yr, ticktext_yr = [], []
for y in unique_years:
    yr_dates = dates_idx[dates_idx.year == y]
    if not yr_dates.empty:
        tickvals_yr.append(min(yr_dates))
        ticktext_yr.append(str(y))

fig_interactive = go.Figure(data=go.Heatmap(
    z=z_log_interactive,
    x=distance_df.columns,
    y=distance_df.index,
    colorscale='gray',
    colorbar=dict(
        title=f'JSD<br>Log Distance',
        tickvals=np.linspace(np.nanmin(z_log_interactive),
                             np.nanmax(z_log_interactive), 5),
        ticktext=[
            f"{10**v:.2f}" for v in
            np.linspace(np.nanmin(z_log_interactive),
                        np.nanmax(z_log_interactive), 5)
        ],
    ),
))

fig_interactive.update_xaxes(
    tickmode='array', tickvals=tickvals_yr, ticktext=ticktext_yr, type='date'
)
fig_interactive.update_yaxes(
    tickmode='array', tickvals=tickvals_yr, ticktext=ticktext_yr, type='date'
)
fig_interactive.update_layout(
    title='',
    xaxis_title='Date', yaxis_title='Date',
    template='plotly_white', width=700, height=700,
)

fig_interactive.show()

## Publication heatmap (Matplotlib) — this is saved

In [ ]:
constant   = 1e-10
mask_diag  = np.eye(distance_df.shape[0], dtype=bool)

z          = distance_df.values.copy()
z_log      = np.log10(np.where(mask_diag, z + constant, z))
z_log_masked = np.ma.masked_array(z_log, mask=mask_diag)

vmin = float(z_log_masked.min())
vmax = float(z_log_masked.max())

x_min = mdates.date2num(distance_df.columns.min())
x_max = mdates.date2num(distance_df.columns.max())
y_min = mdates.date2num(distance_df.index.min())
y_max = mdates.date2num(distance_df.index.max())

fig, ax = plt.subplots(figsize=(7, 7))

cmap_gray = plt.get_cmap('gray').copy()
cmap_gray.set_bad('black')

im = ax.imshow(z_log_masked, aspect='auto', origin='lower', cmap=cmap_gray,
               extent=[x_min, x_max, y_min, y_max], vmin=vmin, vmax=vmax)

# Main colorbar
cbar = fig.colorbar(im, ax=ax, shrink=0.7)
tick_vals_cb = np.linspace(vmin, vmax, 5)
tick_labels_cb = [f"{10**v:.2f}" for v in tick_vals_cb]
tick_labels_cb[0] = "0"
cbar.set_ticks(tick_vals_cb)
cbar.set_ticklabels(tick_labels_cb)
cbar.ax.tick_params(labelsize=10)
cbar.ax.set_title(f'JSD\nDistance', fontsize=10)

ax.xaxis_date()
ax.yaxis_date()

unique_years_ax = [y for y in sorted(set(distance_df.columns.year)) if y != 2008]
tick_dates_ax   = [pd.Timestamp(year=y, month=1, day=1) for y in unique_years_ax]
tick_pos_ax     = [mdates.date2num(t) for t in tick_dates_ax]
tick_lbls_ax    = [t.strftime("%Y") for t in tick_dates_ax]

ax.set_xticks(tick_pos_ax)
ax.set_xticklabels(tick_lbls_ax, fontsize=11, rotation=25)
ax.set_yticks(tick_pos_ax)
ax.set_yticklabels(tick_lbls_ax, fontsize=11)
ax.set_aspect('equal', adjustable='box')

# Viridis colour strips along the top and right to mark the time axis
divider   = make_axes_locatable(ax)
cax_top   = divider.append_axes("top",   size="5%", pad=0.05)
sm_x      = plt.cm.ScalarMappable(cmap='viridis',
                                   norm=mcolors.Normalize(vmin=x_min, vmax=x_max))
sm_x.set_array([])
cbar_top  = plt.colorbar(sm_x, cax=cax_top, orientation='horizontal')
cbar_top.set_ticks([])
cax_top.xaxis.set_ticks_position("top")
cax_top.xaxis.set_label_position("top")

cax_right = divider.append_axes("right", size="5%", pad=0.05)
sm_y      = plt.cm.ScalarMappable(cmap='viridis',
                                   norm=mcolors.Normalize(vmin=y_min, vmax=y_max))
sm_y.set_array([])
cbar_right = plt.colorbar(sm_y, cax=cax_right, orientation='vertical')
cbar_right.set_ticks([])

plt.tight_layout()
plt.show()

## Save figure

In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
fig_dir   = FIGURES_DIR / "Fig.5b_Vectors_matrix" / timestamp
basename  = f"totalfreq_{metric}_{timestamp}"

for fmt in ['pdf', 'svg', 'jpeg']:
    (fig_dir / fmt).mkdir(parents=True, exist_ok=True)

# Publication figure (matplotlib)
fig.savefig(str(fig_dir / "pdf"  / f"{basename}.pdf"),  format='pdf',  bbox_inches='tight')
fig.savefig(str(fig_dir / "jpeg" / f"{basename}.jpg"),  format='jpg',  dpi=500, bbox_inches='tight')
fig.savefig(str(fig_dir / "svg"  / f"{basename}.svg"),  format='svg',  bbox_inches='tight')

# Interactive version (plotly → HTML)
html_path = fig_dir / f"{basename}.html"
fig_interactive.write_html(str(html_path))

print(f"Saved to: {fig_dir}")
print(f"  PDF:  {basename}.pdf")
print(f"  JPEG: {basename}.jpg")
print(f"  SVG:  {basename}.svg")
print(f"  HTML: {basename}.html")